# Lab 4: Build the Audience, Then Buy It
**Name:** Hranko Anna | **Group:** KM-43

## Setup

In [1]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")

programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

Using Colab cache for faster access to the 'adtech-lab4-tv-audiences' dataset.


### Task 1.0 Clean the survey file

In [2]:
va = viewer_attributes.copy()
va['respondent_id'] = va['respondent_id'].astype(str).str.strip()

rl_ids = set(va[va['attr_id'].isin([101, 102])]['respondent_id'])
ages_df = va[va['attr_id'] == 300][['respondent_id', 'value']].copy()
ages_df['age'] = ages_df['value'].astype(float)
adults_df = ages_df[(ages_df['respondent_id'].isin(rl_ids)) & (ages_df['age'] >= 18)].copy()
valid_adult_ids = set(adults_df['respondent_id'])

valid_b = survey_layout[survey_layout['RECTYPE'] == 'B'].copy()
valid_keywords = set(valid_b['KEYWORD'].str.strip().str.upper())

raw_count = len(survey)
df_s = survey.copy()
df_s['RESPID'] = df_s['RESPID'].str.strip()
df_s['KEYWORD'] = df_s['KEYWORD'].str.strip()
df_s['VALUE'] = df_s['VALUE'].str.strip()

bad_records = []

mask_header = (df_s['RESPID'] == 'RESPID') | (df_s['KEYWORD'] == 'KEYWORD')
bad_records.append({"Kind of bad record": "Repeated header row", "Rows": int(mask_header.sum()), "Action": "Dropped"})
df_s = df_s[~mask_header]

yes_vals = {'yes', 'y', '1', '1.0', 'true'}
mask_not_yes = ~df_s['VALUE'].str.lower().isin(yes_vals)
bad_records.append({"Kind of bad record": "Non-yes or blank VALUE", "Rows": int(mask_not_yes.sum()), "Action": "Dropped"})
df_s = df_s[~mask_not_yes]

mask_lower_kw = df_s['KEYWORD'] != df_s['KEYWORD'].str.upper()
df_s['KEYWORD'] = df_s['KEYWORD'].str.upper()
bad_records.append({"Kind of bad record": "Lowercase KEYWORD", "Rows": int(mask_lower_kw.sum()), "Action": "Normalized to uppercase"})

mask_bad_kw = ~df_s['KEYWORD'].isin(valid_keywords)
bad_records.append({"Kind of bad record": "Unknown KEYWORD", "Rows": int(mask_bad_kw.sum()), "Action": "Dropped"})
df_s = df_s[~mask_bad_kw]

orig_id = df_s['RESPID'].copy()
df_s['RESPID'] = df_s['RESPID'].str.replace(r'\.0$', '', regex=True).str.lstrip('0')
mask_malformed_id = orig_id != df_s['RESPID']
bad_records.append({"Kind of bad record": "Malformed RESPID (leading zero or .0)", "Rows": int(mask_malformed_id.sum()), "Action": "Normalized ID format"})

mask_bad_id = ~df_s['RESPID'].isin(valid_adult_ids)
bad_records.append({"Kind of bad record": "Invalid RESPID (under 18, H/S, or unknown)", "Rows": int(mask_bad_id.sum()), "Action": "Dropped"})
df_s = df_s[~mask_bad_id]

mask_dup = df_s.duplicated(subset=['RESPID', 'KEYWORD'], keep='first')
bad_records.append({"Kind of bad record": "Duplicate (RESPID + KEYWORD)", "Rows": int(mask_dup.sum()), "Action": "Deduplicated"})
df_s = df_s[~mask_dup]

pair_kws = ['X710101', 'X710102', 'X710103']
pair_priority = {'X710101': 1, 'X710102': 2, 'X710103': 3}

df_pairs = df_s[df_s['KEYWORD'].isin(pair_kws)].copy()
df_pairs['prio'] = df_pairs['KEYWORD'].map(pair_priority)
people_multi_pairs = df_pairs.groupby('RESPID').filter(lambda g: len(g) > 1)['RESPID'].nunique()

df_pairs_clean = df_pairs.sort_values('prio', ascending=False).drop_duplicates(subset=['RESPID'], keep='first')
dropped_lower_pairs = len(df_pairs) - len(df_pairs_clean)
bad_records.append({"Kind of bad record": "Multiple pair counts per person", "Rows": int(dropped_lower_pairs), "Action": "Kept higher pair count"})

df_s = pd.concat([df_s[~df_s['KEYWORD'].isin(pair_kws)], df_pairs_clean.drop(columns=['prio'])], ignore_index=True)

piv = df_s.assign(flag=1).pivot(index='RESPID', columns='KEYWORD', values='flag').fillna(0)
panel_survey = adults_df[['respondent_id']].rename(columns={'respondent_id': 'RESPID'}).copy()
panel_survey = panel_survey.merge(piv, on='RESPID', how='left').fillna(0)

for k in valid_keywords:
    if k not in panel_survey.columns:
        panel_survey[k] = 0

brand_kws = ['X710201', 'X710202', 'X710203', 'X710298', 'X710299']
has_any_signal = (panel_survey[pair_kws + brand_kws].sum(axis=1) > 0) | (panel_survey['X710100'] > 0)

implied_any = int((has_any_signal & (panel_survey['X710100'] == 0)).sum())
bad_records.append({"Kind of bad record": "Buyer missing explicit Any (X710100) row", "Rows": implied_any, "Action": "Imputed Any = 1"})

panel_survey['buyer'] = np.where(has_any_signal, 'yes', 'no')
panel_survey['X710100'] = np.where(has_any_signal, 1, 0)

def get_pairs(row):
    if row['X710103'] == 1:
        return '3+'
    elif row['X710102'] == 1:
        return '2'
    elif row['X710101'] == 1:
        return '1'
    return '0'

panel_survey['pairs'] = panel_survey.apply(get_pairs, axis=1)

check_df = valid_b[['KEYWORD', 'Text', 'SAMPLE']].copy()
check_df['Your_Count'] = check_df['KEYWORD'].map(lambda k: int(panel_survey[k].sum()))
check_df['Diff'] = check_df['Your_Count'] - check_df['SAMPLE'].astype(int)

print("Survey data rows:", raw_count)
print("Clean yes answers:", int(panel_survey[list(valid_keywords)].sum().sum()))
print("Adult panel members:", len(panel_survey))
print("Non-buyers:", int((panel_survey['buyer'] == 'no').sum()))
print("People with two pair counts:", people_multi_pairs)

display(pd.DataFrame(bad_records))
display(check_df)

raw_kestrel = survey[survey['KEYWORD'].str.strip().str.upper() == 'X710202']['RESPID'].nunique()
clean_kestrel = int(panel_survey['X710202'].sum())
raw_buyers_share = survey[survey['KEYWORD'].str.strip().str.upper() == 'X710100']['RESPID'].nunique() / len(panel_survey)
clean_buyers_share = (panel_survey['buyer'] == 'yes').mean()
print(f"Raw Kestrel count: {raw_kestrel} vs Clean: {clean_kestrel} (diff: {raw_kestrel - clean_kestrel})")
print(f"Raw buyer share: {raw_buyers_share:.3f} vs Clean: {clean_buyers_share:.3f} (diff: {raw_buyers_share - clean_buyers_share:.3f})")

Survey data rows: 18349
Clean yes answers: 15635
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113


,Kind of bad record,Rows,Action
0,Repeated header row,2,Dropped
1,Non-yes or blank VALUE,743,Dropped
2,Lowercase KEYWORD,706,Normalized to uppercase
3,Unknown KEYWORD,738,Dropped
4,Malformed RESPID (leading zero or .0),816,Normalized ID format
5,"Invalid RESPID (under 18, H/S, or unknown)",423,Dropped
6,Duplicate (RESPID + KEYWORD),933,Deduplicated
7,Multiple pair counts per person,113,Kept higher pair count
8,Buyer missing explicit Any (X710100) row,238,Imputed Any = 1


,KEYWORD,Text,SAMPLE,Your_Count,Diff
2,X710100,Any,4779,4779,0
3,X710101,1 pair,1810,1810,0
4,X710102,2 pairs,1308,1308,0
5,X710103,3 or more pairs,1661,1661,0
7,X710201,STRIDE,1563,1563,0
8,X710202,KESTREL,354,354,0
9,X710203,EASYWALK,1050,1050,0
10,X710298,OTHER BRAND,2739,2739,0
11,X710299,DON'T KNOW,371,371,0


Raw Kestrel count: 530 vs Clean: 354 (diff: 176)
Raw buyer share: 0.585 vs Clean: 0.597 (diff: -0.012)


**Answer:**
Looking through the raw survey file, I found several data quality issues: repeated header rows inside the file, "no" or blank entries instead of only "yes" responses, lowercase keywords with trailing spaces, unknown codes not in the layout, IDs formatted with leading zeros or .0 endings, rows for under-18 or household/guest records, exact duplicates, people who reported two different pair counts (where I kept the higher count), and buyers missing the general Any (X710100) row. If we had used the raw file without cleaning it first, the Kestrel buyer count would have been off by 176 people (530 instead of 354) and the overall buyer share would have looked different because "no" rows, duplicates, and non-adult IDs would have been mixed in while formatted IDs and lowercase keywords were missed.

### Task 1.1 Four definitions

In [3]:
sex_df = va[va['attr_id'].isin([201, 202])][['respondent_id', 'attr_id']].drop_duplicates('respondent_id')
sex_df['sex'] = np.where(sex_df['attr_id'] == 201, 'Men', 'Women')

adults_full = adults_df[['respondent_id', 'age']].merge(sex_df[['respondent_id', 'sex']], on='respondent_id', how='left')
adults_full = adults_full.rename(columns={'respondent_id': 'RESPID'}).merge(panel_survey, on='RESPID', how='left')

adults_full['aud_adults'] = True
adults_full['aud_buyers'] = adults_full['buyer'] == 'yes'
adults_full['aud_heavy'] = adults_full['pairs'] == '3+'
adults_full['aud_kestrel'] = adults_full['X710202'] == 1
adults_full['aud_25_54'] = (adults_full['age'] >= 25) & (adults_full['age'] <= 54)

dw = daily_weights[daily_weights['resp_type'].isin(['R', 'L'])].copy()
dw['respondent_id'] = dw['respondent_id'].astype(str).str.strip()
dw_adults = dw.merge(adults_full, left_on='respondent_id', right_on='RESPID', how='inner')
n_days = dw_adults['broadcast_date'].nunique()

def calc_aud_size(col_name):
    return dw_adults[dw_adults[col_name]].groupby('broadcast_date')['weight'].sum().sum() / n_days

size_adults = calc_aud_size('aud_adults')
size_buyers = calc_aud_size('aud_buyers')
size_heavy = calc_aud_size('aud_heavy')
size_kestrel = calc_aud_size('aud_kestrel')
size_25_54 = calc_aud_size('aud_25_54')

print(f"Adults 18+ (M): {size_adults / 1e6:.1f}")
print(f"Shoe buyers (M): {size_buyers / 1e6:.1f}")
print(f"Heavy buyers, 3+ pairs (M): {size_heavy / 1e6:.1f}")
print(f"Kestrel buyers (M): {size_kestrel / 1e6:.1f}")
print(f"Adults 25-54 (M): {size_25_54 / 1e6:.1f}")
print(f"Kestrel buyers in panel (n): {int(adults_full['aud_kestrel'].sum())}")

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


### Task 1.2 Who are they?

In [4]:
def age_grp(a):
    if a <= 34:
        return '18-34'
    elif a <= 54:
        return '35-54'
    return '55+'

dw_adults['demo_6'] = dw_adults['sex'] + ' ' + dw_adults['age'].apply(age_grp)
groups_order = ['Men 18-34', 'Men 35-54', 'Men 55+', 'Women 18-34', 'Women 35-54', 'Women 55+']
aud_map = {
    'Adults 18+': 'aud_adults',
    'Shoe buyers': 'aud_buyers',
    'Heavy buyers (3+)': 'aud_heavy',
    'Kestrel buyers': 'aud_kestrel',
    'Adults 25-54': 'aud_25_54'
}

base_shares = {}
for g in groups_order:
    g_size = dw_adults[dw_adults['demo_6'] == g].groupby('broadcast_date')['weight'].sum().sum() / n_days
    base_shares[g] = g_size / size_adults

rows_12 = []
for aud_label, col in aud_map.items():
    aud_tot = calc_aud_size(col)
    row_dict = {'Audience': aud_label}
    for g in groups_order:
        sub = dw_adults[dw_adults[col] & (dw_adults['demo_6'] == g)]
        g_size = sub.groupby('broadcast_date')['weight'].sum().sum() / n_days
        share = g_size / aud_tot
        idx = (share / base_shares[g]) * 100
        row_dict[f"{g} Share"] = round(share, 3)
        row_dict[f"{g} Index"] = round(idx, 1)
    rows_12.append(row_dict)

display(pd.DataFrame(rows_12))

,Audience,Men 18-34 Share,Men 18-34 Index,Men 35-54 Share,Men 35-54 Index,Men 55+ Share,Men 55+ Index,Women 18-34 Share,Women 18-34 Index,Women 35-54 Share,Women 35-54 Index,Women 55+ Share,Women 55+ Index
0,Adults 18+,0.144,100.0,0.160,100.0,0.185,100.0,0.144,100.0,0.160,100.0,0.208,100.0
1,Shoe buyers,0.140,97.4,0.158,99.2,0.186,100.8,0.140,97.6,0.178,111.4,0.197,94.6
2,Heavy buyers (3+),0.111,77.1,0.138,86.8,0.189,102.2,0.139,97.0,0.226,141.9,0.196,94.0
3,Kestrel buyers,0.118,82.1,0.189,118.6,0.126,68.2,0.181,126.0,0.265,165.8,0.121,58.0
4,Adults 25-54,0.174,121.2,0.326,204.1,0.000,0.0,0.174,121.2,0.326,204.1,0.000,0.0


**Answer:**
I would hand the planner the Heavy buyers (3+ pairs) definition. It matches the client's brief, gives a big enough audience to plan around, and has a solid panel sample size, whereas Kestrel buyers only has 354 people in the panel (way too noisy for program ratings) and limits the campaign to existing customers instead of winning buyers from competitors. Athletic-shoe buyers looks almost identical to all adults because most people buy at least one pair of sneakers a year, so the category is too broad to separate heavy shoppers from everyone else. One thing a fused purchase attribute cannot tell you is whether people inside the same demographic group actually watch different shows—for example, whether serious runners watch sports broadcasts more than non-runners of the same age, sex, and income.

### Task 2.1 Programs by audience

In [5]:
prime_progs = programs[
    (programs['daypart'].str.lower() == 'prime') &
    (programs['broadcast_date'].astype(str).str.startswith('2024-10'))
].copy()

prime_progs['start_dt'] = pd.to_datetime(prime_progs['start_ts'])
prime_progs['end_dt'] = pd.to_datetime(prime_progs['end_ts'])

v_rl = viewings[viewings['resp_type'].isin(['R', 'L'])].drop(columns=['weight'], errors='ignore').copy()
v_rl['respondent_id'] = v_rl['respondent_id'].astype(str).str.strip()
v_rl['v_start'] = pd.to_datetime(v_rl['start_ts'])
v_rl['v_end'] = pd.to_datetime(v_rl['end_ts'])

v_rl = v_rl.merge(
    adults_full[['RESPID', 'aud_adults', 'aud_heavy', 'aud_kestrel']],
    left_on='respondent_id',
    right_on='RESPID',
    how='inner'
)

target_shows = [
    ("Sunday Night Football", "NFL Football", "NBC", "NBC"),
    ("Monday Night Football", "NFL Football", "ABC", "ABC"),
    ("Dancing with the Stars", "Dancing with the Stars", "ABC", "ABC"),
    ("NCIS", "NCIS", "CBS", "CBS"),
    ("Hannity", "Hannity", "Fox News Channel", "Fox News"),
    ("Unsellable Houses", "Unsellable Houses", "HGTV", "HGTV"),
    ("Halloween Wars", "Halloween Wars", "Food Network", "Food Network"),
    ("Grotesquerie", "Grotesquerie", "FX", "FX")
]

target_telecasts = []
for label_show, prog_db, net_db, label_net in target_shows:
    sub = prime_progs[
        (prime_progs['program_name'].str.lower() == prog_db.lower()) &
        (prime_progs['network_name'].str.lower() == net_db.lower())
    ].copy()
    sub['display_label'] = f"{label_show} ({label_net})"
    target_telecasts.append(sub)
target_tc_df = pd.concat(target_telecasts, ignore_index=True)

v_joined = v_rl.merge(
    target_tc_df[['telecast_id', 'network_code', 'broadcast_date', 'start_dt', 'end_dt', 'duration_min']],
    on='network_code',
    how='inner'
)

overlap_start = np.maximum(v_joined['v_start'], v_joined['start_dt'])
overlap_end = np.minimum(v_joined['v_end'], v_joined['end_dt'])
v_joined['overlap_min'] = (overlap_end - overlap_start).dt.total_seconds() / 60.0
v_joined = v_joined[v_joined['overlap_min'] > 0].copy()

v_joined['broadcast_date'] = v_joined['broadcast_date'].astype(str)
dw_adults['broadcast_date'] = dw_adults['broadcast_date'].astype(str)

v_joined = v_joined.merge(
    dw_adults[['RESPID', 'broadcast_date', 'weight']],
    on=['RESPID', 'broadcast_date'],
    how='inner'
)

v_joined['w_min_adults'] = v_joined['weight'] * v_joined['overlap_min'] * v_joined['aud_adults']
v_joined['w_min_heavy'] = v_joined['weight'] * v_joined['overlap_min'] * v_joined['aud_heavy']
v_joined['w_min_kestrel'] = v_joined['weight'] * v_joined['overlap_min'] * v_joined['aud_kestrel']

tc_agg = v_joined.groupby('telecast_id')[['w_min_adults', 'w_min_heavy', 'w_min_kestrel']].sum().reset_index()
tc_eval = target_tc_df.merge(tc_agg, on='telecast_id', how='left').fillna({'w_min_adults': 0, 'w_min_heavy': 0, 'w_min_kestrel': 0})

daily_adults = dw_adults[dw_adults['aud_adults']].groupby('broadcast_date')['weight'].sum().to_dict()
daily_heavy = dw_adults[dw_adults['aud_heavy']].groupby('broadcast_date')['weight'].sum().to_dict()
daily_kestrel = dw_adults[dw_adults['aud_kestrel']].groupby('broadcast_date')['weight'].sum().to_dict()

tc_eval['b_date_str'] = tc_eval['broadcast_date'].astype(str)
tc_eval['sz_adults'] = tc_eval['b_date_str'].map(daily_adults)
tc_eval['sz_heavy'] = tc_eval['b_date_str'].map(daily_heavy)
tc_eval['sz_kestrel'] = tc_eval['b_date_str'].map(daily_kestrel)

tc_eval['rtg_adults'] = (tc_eval['w_min_adults'] / (tc_eval['duration_min'] * tc_eval['sz_adults'])) * 100
tc_eval['rtg_heavy'] = (tc_eval['w_min_heavy'] / (tc_eval['duration_min'] * tc_eval['sz_heavy'])) * 100
tc_eval['rtg_kestrel'] = (tc_eval['w_min_kestrel'] / (tc_eval['duration_min'] * tc_eval['sz_kestrel'])) * 100

tc_eval['imp_adults'] = tc_eval['w_min_adults'] / tc_eval['duration_min']
tc_eval['imp_heavy'] = tc_eval['w_min_heavy'] / tc_eval['duration_min']

def get_cost_30s(row):
    rc_net = rate_card[rate_card['network_code'] == row['network_code']]
    prog_rows = rc_net[rc_net['program_name'].fillna('').str.lower() == str(row['program_name']).lower()]
    if len(prog_rows) > 0:
        return float(prog_rows.iloc[0]['cost_30s_usd'])
    wd = pd.to_datetime(row['broadcast_date']).weekday()
    dp_rows = rc_net[rc_net['daypart'].fillna('').str.contains('prime', case=False, na=False)]
    for _, r_rc in dp_rows.iterrows():
        if r_rc['weekdays'] is not None and wd in list(r_rc['weekdays']):
            return float(r_rc['cost_30s_usd'])
    if len(dp_rows) > 0:
        return float(dp_rows.iloc[0]['cost_30s_usd'])
    return float(rc_net.iloc[0]['cost_30s_usd'])

tc_eval['cost_30s'] = tc_eval.apply(get_cost_30s, axis=1)

rows_21 = []
for label_show, prog_db, net_db, label_net in target_shows:
    lbl = f"{label_show} ({label_net})"
    sub = tc_eval[tc_eval['display_label'] == lbl]
    rtg_a = sub['rtg_adults'].mean()
    rtg_t = sub['rtg_heavy'].mean()
    rtg_k = sub['rtg_kestrel'].mean()
    imp_a = sub['imp_adults'].mean()
    imp_t = sub['imp_heavy'].mean()
    cost_m = sub['cost_30s'].mean()

    comp = imp_t / imp_a if imp_a > 0 else 0
    idx_h = (rtg_t / rtg_a) * 100 if rtg_a > 0 else 0
    idx_k = (rtg_k / rtg_a) * 100 if rtg_a > 0 else 0
    cpm_a = (cost_m / imp_a) * 1000 if imp_a > 0 else np.nan
    cpm_t = (cost_m / imp_t) * 1000 if imp_t > 0 else np.nan

    rows_21.append({
        'Program': lbl,
        'Adults rating': round(rtg_a, 2),
        'Target rating': round(rtg_t, 2),
        'Target viewers (k)': round(imp_t / 1000, 1),
        'Composition': round(comp, 3),
        'Index': round(idx_h, 1),
        'Cost per :30': int(round(cost_m)),
        'Adults CPM': round(cpm_a, 2),
        'Target CPM': round(cpm_t, 2),
        'Kestrel index': round(idx_k, 1)
    })

df_21 = pd.DataFrame(rows_21)
display(df_21.drop(columns=['Kestrel index']))
print(f"Target universe (M): {size_heavy / 1e6:.1f}")

,Program,Adults rating,Target rating,Target viewers (k),Composition,Index,Cost per :30,Adults CPM,Target CPM
0,Sunday Night Football (NBC),5.82,5.79,2865.0,0.195,99.3,487200,33.12,170.05
1,Monday Night Football (ABC),8.93,8.36,4149.1,0.184,93.6,723600,32.09,174.40
2,Dancing with the Stars (ABC),3.25,3.14,1564.2,0.191,96.8,94333,11.49,60.31
3,NCIS (CBS),2.21,2.43,1206.3,0.216,110.1,58100,10.43,48.16
4,Hannity (Fox News),1.09,0.84,416.9,0.152,77.1,39900,14.50,95.71
5,Unsellable Houses (HGTV),0.50,0.64,317.2,0.249,126.8,12900,10.12,40.66
6,Halloween Wars (Food Network),0.28,0.33,161.9,0.228,116.2,9700,13.67,59.90
7,Grotesquerie (FX),0.26,0.25,124.2,0.188,95.9,8000,12.11,64.39


Target universe (M): 49.6


### Task 2.2 Index, volume or price?

In [6]:
best_idx_prog = df_21.loc[df_21['Index'].idxmax(), 'Program']
best_vol_prog = df_21.loc[df_21['Target viewers (k)'].idxmax(), 'Program']
best_cpm_prog = df_21.loc[df_21['Target CPM'].idxmin(), 'Program']

print("Highest index (program):", best_idx_prog)
print("Most target viewers (program):", best_vol_prog)
print("Lowest target CPM (program):", best_cpm_prog)

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): Monday Night Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:**
No, they are not the same program: Unsellable Houses (HGTV) has the highest index (126.8) and the lowest target CPM ($40.66), while Monday Night Football (ABC) delivers the most target viewers (4,149.1k). If a client asks to buy only the highest index, I would use the shampoo example from Lecture 5 to explain that index only tells you how concentrated the target audience is, not how many people you actually reach or what the spot costs. A high-index cable show can have tiny reach so you would have to buy tons of spots to build scale, and in general a high-index show could still be overpriced if its base rate is high, so looking at target CPM alongside total volume is a much smarter way to spend the budget.

### Task 2.3 Kestrel buyers

In [7]:
display(df_21[['Program', 'Index', 'Kestrel index']].rename(columns={'Index': 'Heavy-buyer index'}))

,Program,Heavy-buyer index,Kestrel index
0,Sunday Night Football (NBC),99.3,98.5
1,Monday Night Football (ABC),93.6,85.9
2,Dancing with the Stars (ABC),96.8,82.7
3,NCIS (CBS),110.1,56.5
4,Hannity (Fox News),77.1,42.6
5,Unsellable Houses (HGTV),126.8,76.9
6,Halloween Wars (Food Network),116.2,95.2
7,Grotesquerie (FX),95.9,129.6


**Answer:**
The Kestrel buyer index jumps around much more than the heavy-buyer index because there are only 354 Kestrel buyers in the entire panel (compared to 1,661 heavy buyers). When a niche cable show already has a small audience, just one or two Kestrel panelists tuning in or out completely swings its rating and index. I would trust the heavy-buyer index because its larger sample size keeps sampling error much lower. To check if an index is actually reliable, we could calculate confidence intervals (standard errors) around the ratings or split October into separate weeks to see if the program's index stays steady week to week.

### Task 3.1 Smart TV weights

In [8]:
ip_counts = hh_attributes['hashed_ip'].value_counts()
valid_ips = set(ip_counts[ip_counts == 1].index)

hh_one = hh_attributes[hh_attributes['hashed_ip'].isin(valid_ips)].copy()
hh_one['head_age'] = np.where(hh_one['head_age_band'] == '65+', '65+', 'Under 65')

acr_m = acr_sessions[acr_sessions['hashed_ip'].isin(valid_ips)].copy()
acr_m['s_start_et'] = pd.to_datetime(acr_m['start_utc'], utc=True).dt.tz_convert('America/New_York')
acr_m['s_end_et'] = pd.to_datetime(acr_m['end_utc'], utc=True).dt.tz_convert('America/New_York')
acr_m['dur_sec'] = (acr_m['s_end_et'] - acr_m['s_start_et']).dt.total_seconds()
acr_m = acr_m[acr_m['dur_sec'] >= 60].copy()

tv_hh_map = acr_m[['tv_id', 'hashed_ip']].drop_duplicates('tv_id').merge(
    hh_one[['hashed_ip', 'head_age', 'income_band']], on='hashed_ip', how='inner'
)
n_matched_tvs = tv_hh_map['tv_id'].nunique()

acr_m['b_date'] = (acr_m['s_start_et'] - pd.Timedelta(hours=6)).dt.strftime('%Y-%m-%d')
intab_tv_day = acr_m[['tv_id', 'b_date']].drop_duplicates().merge(tv_hh_map, on='tv_id', how='inner')

cell_day_cnt = intab_tv_day.groupby(['b_date', 'head_age', 'income_band'])['tv_id'].nunique().reset_index(name='n_tvs')
w_day = cell_day_cnt.merge(universe, on=['head_age', 'income_band'], how='inner')
w_day['w'] = w_day['tv_households'] / w_day['n_tvs']

intab_weighted = intab_tv_day.merge(w_day[['b_date', 'head_age', 'income_band', 'w']], on=['b_date', 'head_age', 'income_band'], how='inner')
weights_sum_day = intab_weighted.groupby('b_date')['w'].sum().mean()

ref_progs = programs.merge(reference_ratings, on='telecast_id', how='inner').copy()
ref_progs['p_start_et'] = pd.to_datetime(ref_progs['start_ts']).dt.tz_localize('America/New_York')
ref_progs['p_end_et'] = pd.to_datetime(ref_progs['end_ts']).dt.tz_localize('America/New_York')
ref_progs['b_date'] = (ref_progs['p_start_et'] - pd.Timedelta(hours=6)).dt.strftime('%Y-%m-%d')

acr_lin = acr_m[acr_m['session_type'].isin(['live', 'dvr']) & acr_m['network_code'].notna()].copy()
acr_j = acr_lin.merge(
    ref_progs[['telecast_id', 'program_name', 'network_name', 'network_code', 'b_date', 'p_start_et', 'p_end_et', 'duration_min', 'hh_rating']],
    on=['network_code', 'b_date'],
    how='inner'
)

ov_s = np.maximum(acr_j['s_start_et'], acr_j['p_start_et'])
ov_e = np.minimum(acr_j['s_end_et'], acr_j['p_end_et'])
acr_j['ov_min'] = (ov_e - ov_s).dt.total_seconds() / 60.0
acr_j = acr_j[acr_j['ov_min'] >= 1.0].copy()

tv_tc = acr_j.groupby(['telecast_id', 'b_date', 'tv_id', 'duration_min'])['ov_min'].sum().reset_index()
tv_tc['ov_min'] = np.minimum(tv_tc['ov_min'], tv_tc['duration_min'])
tv_tc = tv_tc.merge(intab_weighted[['b_date', 'tv_id', 'w']], on=['b_date', 'tv_id'], how='inner')

rows_31 = []
for _, r in ref_progs.iterrows():
    tid = r['telecast_id']
    bd = r['b_date']
    dur = r['duration_min']
    sub_intab = intab_weighted[intab_weighted['b_date'] == bd]
    n_intab = len(sub_intab)
    w_intab = sub_intab['w'].sum()

    sub_w = tv_tc[tv_tc['telecast_id'] == tid]
    unw_rtg = (sub_w['ov_min'].sum() / (dur * n_intab)) * 100
    w_rtg = ((sub_w['ov_min'] * sub_w['w']).sum() / (dur * w_intab)) * 100

    rows_31.append({
        'telecast': f"{r['program_name']} ({r['network_name']})",
        'unweighted': round(unw_rtg, 2),
        'weighted': round(w_rtg, 2),
        'reference': round(r['hh_rating'], 2)
    })

df_31 = pd.DataFrame(rows_31)
gap_unw = (df_31['reference'] - df_31['unweighted']).abs().mean()
gap_w = (df_31['reference'] - df_31['weighted']).abs().mean()
share_closed = (gap_unw - gap_w) / gap_unw * 100

print(f"TVs matched to one household: {n_matched_tvs}")
print(f"Weights sum per day (M): {weights_sum_day / 1e6:.1f}")
display(df_31)
print(f"Mean gap unweighted (points): {gap_unw:.2f}")
print(f"Mean gap weighted (points): {gap_w:.2f}")
print(f"Share of gap closed: {share_closed:.1f}%")

TVs matched to one household: 19607
Weights sum per day (M): 125.8


,telecast,unweighted,weighted,reference
0,The Real Housewives of New York City (Bravo),0.24,0.22,1.23
1,NCIS (CBS),2.28,2.43,10.56
2,Hannity (Fox News Channel),0.73,0.94,2.30
3,Halloween Baking Championship (Food Network),0.38,0.36,1.63
4,MLB Postseason (FOX),3.96,4.18,25.86
5,Grotesquerie (FX),0.64,0.62,3.53
6,Unsellable Houses (HGTV),0.75,0.73,3.69
7,NFL Football (NBC),5.36,5.72,33.14


Mean gap unweighted (points): 8.45
Mean gap weighted (points): 8.34
Share of gap closed: 1.3%


### Task 3.2 What weighting cannot fix

**Answer:**
The ratings went up a little bit on older shows like Hannity (from 0.73 to 0.94), NCIS (2.28 to 2.43), NFL Football (5.36 to 5.72), and MLB Postseason (3.96 to 4.18). At the same time, they went down slightly on younger shows like The Real Housewives of New York City (0.24 to 0.22) and Unsellable Houses (0.75 to 0.73). This happens because smart TV owners have fewer 65+ households, so weighting gives those older homes more weight. However, weighting only closes 1.3% of the gap. Most of the difference stays because weights cannot fix technical limits: ACR only tracks one smart TV in the house, misses linear channels watched through external HDMI boxes or apps, and only includes people who turned ACR on. Income weights barely change the numbers because income does not affect TV show choices as much as age, and our sample already has a pretty normal income mix.

### Task 4.1 October predicts October

In [9]:
import gc

heavy_ids = set(adults_full.loc[adults_full['aud_heavy'], 'RESPID'])

dw_p2 = daily_weights[daily_weights['resp_type'].isin(['R', 'L'])][['respondent_id', 'broadcast_date', 'weight']].copy()
dw_p2['RESPID'] = dw_p2['respondent_id'].astype(str).str.strip()
dw_p2['b_date'] = dw_p2['broadcast_date'].astype(str)
dw_p2['is_heavy'] = dw_p2['RESPID'].isin(heavy_ids)

daily_sz_p2 = dw_p2.groupby('b_date')['weight'].sum().to_dict()
daily_sz_hv = dw_p2[dw_p2['is_heavy']].groupby('b_date')['weight'].sum().to_dict()

nets_4 = ['CBS', 'NBC', 'ABC', 'FOX', 'Fox News Channel', 'Fox News']
prime_4 = programs[
    (programs['daypart'].str.lower() == 'prime') &
    (programs['network_name'].isin(nets_4)) &
    (programs['broadcast_date'].astype(str) >= '2024-10-01') &
    (programs['broadcast_date'].astype(str) <= '2024-10-28')
].copy()
prime_4['b_date'] = prime_4['broadcast_date'].astype(str)
prime_4['p_start'] = pd.to_datetime(prime_4['start_ts'])
prime_4['p_end'] = pd.to_datetime(prime_4['end_ts'])

target_nets = set(prime_4['network_code'].unique())
v_4 = viewings[
    (viewings['resp_type'].isin(['R', 'L'])) &
    (viewings['network_code'].isin(target_nets))
][['respondent_id', 'network_code', 'start_ts', 'end_ts']].copy()

v_4['RESPID'] = v_4['respondent_id'].astype(str).str.strip()
v_4['v_start'] = pd.to_datetime(v_4['start_ts'])
v_4['v_end'] = pd.to_datetime(v_4['end_ts'])
v_4['b_date'] = (v_4['v_start'] - pd.Timedelta(hours=6)).dt.strftime('%Y-%m-%d')
v_4 = v_4[v_4['b_date'].isin(daily_sz_p2.keys())].drop(columns=['respondent_id', 'start_ts', 'end_ts'])

agg_list = []
for bd, p_day in prime_4.groupby('b_date'):
    v_day = v_4[v_4['b_date'] == bd]
    if v_day.empty:
        continue
    m = v_day.merge(p_day[['telecast_id', 'network_code', 'p_start', 'p_end']], on='network_code', how='inner')
    ov_s = np.maximum(m['v_start'], m['p_start'])
    ov_e = np.minimum(m['v_end'], m['p_end'])
    m['ov_min'] = (ov_e - ov_s).dt.total_seconds() / 60.0
    m = m[m['ov_min'] > 0]
    if m.empty:
        continue
    dw_day = dw_p2[dw_p2['b_date'] == bd][['RESPID', 'weight', 'is_heavy']]
    m = m.merge(dw_day, on='RESPID', how='inner')
    m['w_p2'] = m['weight'] * m['ov_min']
    m['w_hv'] = m['w_p2'] * m['is_heavy']
    agg_list.append(m.groupby('telecast_id')[['w_p2', 'w_hv']].sum().reset_index())

del v_4
gc.collect()

agg_4 = pd.concat(agg_list, ignore_index=True).groupby('telecast_id')[['w_p2', 'w_hv']].sum().reset_index()
tc_4 = prime_4.merge(agg_4, on='telecast_id', how='left').fillna({'w_p2': 0, 'w_hv': 0})

tc_4['sz_p2'] = tc_4['b_date'].map(daily_sz_p2)
tc_4['sz_hv'] = tc_4['b_date'].map(daily_sz_hv)
tc_4['rtg_p2'] = (tc_4['w_p2'] / (tc_4['duration_min'] * tc_4['sz_p2'])) * 100
tc_4['rtg_hv'] = (tc_4['w_hv'] / (tc_4['duration_min'] * tc_4['sz_hv'])) * 100

hist_4 = tc_4[tc_4['b_date'] <= '2024-10-21']
test_4 = tc_4[tc_4['b_date'] >= '2024-10-22'].copy()

hist_means = hist_4.groupby(['network_code', 'program_name'])[['rtg_p2', 'rtg_hv']].mean().reset_index().rename(
    columns={'rtg_p2': 'fc_p2', 'rtg_hv': 'fc_hv'}
)

fc_eval = test_4.merge(hist_means, on=['network_code', 'program_name'], how='inner')
fc_eval = fc_eval[(fc_eval['rtg_p2'] > 0) & (fc_eval['rtg_hv'] > 0)].copy()

fc_eval['ape_p2'] = (fc_eval['rtg_p2'] - fc_eval['fc_p2']).abs() / fc_eval['rtg_p2'] * 100
fc_eval['ape_hv'] = (fc_eval['rtg_hv'] - fc_eval['fc_hv']).abs() / fc_eval['rtg_hv'] * 100

fc_eval['net_label'] = fc_eval['network_name'].replace({'Fox News Channel': 'Fox News'})
fc_eval['prog_label'] = fc_eval['program_name'] + ' (' + fc_eval['net_label'] + ')'

biggest_idx = fc_eval['ape_p2'].idxmax()
biggest_row = fc_eval.loc[biggest_idx]

print(f"Telecasts forecast (n): {len(fc_eval)}")
print(f"MAPE, P2+ (%): {fc_eval['ape_p2'].mean():.1f}")
print(f"MAPE, heavy buyers (%): {fc_eval['ape_hv'].mean():.1f}")
print(f"Biggest miss, P2+ (program, date): {biggest_row['prog_label']}, {biggest_row['b_date']}")

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX), 2024-10-27


### Task 4.2 Why it missed

In [14]:
top5_misses = fc_eval.sort_values('ape_p2', ascending=False).head(5).copy()

tbl_42 = pd.DataFrame({
    'program': top5_misses['prog_label'],
    'date': top5_misses['b_date'],
    'forecast': top5_misses['fc_p2'].round(2),
    'actual': top5_misses['rtg_p2'].round(2)
}).reset_index(drop=True)

display(tbl_42)

,program,date,forecast,actual
0,Krapopolis (FOX),2024-10-27,1.06,0.22
1,Universal Basic Guys (FOX),2024-10-27,0.68,0.26
2,Fire Country (CBS),2024-10-27,1.70,0.67
3,Accused (FOX),2024-10-22,0.95,0.37
4,What Would You Do? (ABC),2024-10-23,0.44,0.18


**Answer:**
Most of the big misses happened on Sunday, October 27, when late NFL games pushed the evening schedule back and big sports games on NBC took viewers away. On CBS, football ran late and moved Fire Country to 22:44 instead of its normal time, so its rating dropped from the 1.70 forecast to 0.67. FOX also shifted its schedule and showed late repeats, which lowered Krapopolis (0.22 vs. 1.06) and Universal Basic Guys (0.26 vs. 0.68). Since MAPE divides the error by the actual rating, small drops on smaller shows like Accused and What Would You Do? turn into huge percentages. The heavy-buyer forecast is less accurate (38.6% vs. 35.8% MAPE) because heavy shoe buyers are a much smaller group in the panel than all P2+ viewers, so the numbers jump around more. A better forecast would use the exact broadcast hour and lead-in show, and also check for big live sports on other channels.

### Task 5.1 The plan

In [15]:
spots_plan = {
    'NCIS (CBS)': 8,
    'Unsellable Houses (HGTV)': 20,
    'Dancing with the Stars (ABC)': 2,
    'Halloween Wars (Food Network)': 9
}

plan_df = df_21[df_21['Program'].isin(spots_plan.keys())].copy()
plan_df['spots'] = plan_df['Program'].map(spots_plan)
plan_df['cost'] = (plan_df['spots'] * plan_df['Cost per :30']).round().astype(int)
plan_df['target impressions'] = (plan_df['spots'] * plan_df['Target viewers (k)'] * 1000).round().astype(int)

out_51 = plan_df[['Program', 'spots', 'cost', 'target impressions']].rename(columns={'Program': 'program'}).reset_index(drop=True)
display(out_51)

total_cost = out_51['cost'].sum()
total_target_imp = out_51['target impressions'].sum()

print(f"Plan cost ($): {total_cost}")
print(f"Plan target impressions (M): {total_target_imp / 1e6:.1f}")

,program,spots,cost,target impressions
0,Dancing with the Stars (ABC),2,188666,3128400
1,NCIS (CBS),8,464800,9650400
2,Unsellable Houses (HGTV),20,258000,6344000
3,Halloween Wars (Food Network),9,87300,1457100


Plan cost ($): 998766
Plan target impressions (M): 20.6


### Task 5.2 Recommendation

**Answer:**
We recommend spending \$998,766 across four shows: 20 spots on Unsellable Houses (HGTV), 8 spots on NCIS (CBS), 9 spots on Halloween Wars (Food Network), and 2 spots on Dancing with the Stars (ABC) to deliver 20.6M target impressions. We skipped Sunday Night Football and Monday Night Football because their target CPMs are over \$170, while Unsellable Houses (\$40.66) and NCIS (\$48.16) give us much lower cost per target viewer plus strong heavy-buyer indices (126.8 and 110.1). Adding Dancing with the Stars helps us keep high weekly reach on a major broadcast network at a reasonable \$60.31 target CPM. Because our Task 4 forecast had a 38.6% MAPE for heavy buyers, delivery is not guaranteed and we should ask the networks for guaranteed impressions with make-good spots if ratings fall short. Even with this targeted buy, the blended composition is about 22.1%, which means roughly 78% of the money still reaches adults who are not heavy shoe buyers. After the November flight ends, we should run a post-buy report using panel viewing data to check actual heavy-buyer impressions against the guarantee and track any lift in Kestrel store or website traffic.